# GRAIL — CAL1 human agreement: server steps

Run the cells **in order, one at a time**, and read the output before the next one.
Nothing here deletes data. Cells 1–4 are safe to rerun.

Before cell 5 (the first insurance answers) the pre-registration must be signed,
or the professor's OK must be in writing.

On the Mac first: sign the insurance checklist and push (see the chat / runbook).

In [ ]:
# 1. SETUP AND LOOK — changes nothing
import subprocess, os, getpass, time, hashlib, datetime
def sh(c, t=1800):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=t)
    return (r.stdout + r.stderr).strip()

W    = "/home/jovyan/work"
VENV = "/home/jovyan/vault/grail-cache/venv"
PY   = f"{VENV}/bin/python"
HF   = "/home/jovyan/vault/grail-cache/hf"
REPO = "https://github.com/prasadwatane/CASE-STUDY.git"
ENV  = {**os.environ, "HF_HOME": HF}

# the [r] trick stops pgrep from matching this cell's own shell
print("running jobs :", sh("pgrep -af '[r]un_models|[r]un_probes|[r]un_judge'") or "none")
print("gpu          :", sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader") or "NONE")
print("vault venv   :", sh(f"{PY} -c 'import torch,vllm;print(\"cuda\",torch.cuda.is_available(),\"| vllm\",vllm.__version__)'")
      if os.path.isfile(PY) else "MISSING -> run: python3 /home/jovyan/work/scripts/bootstrap_server.py")
print("repo cloned  :", os.path.isdir(f"{W}/.git"))
if os.path.isdir(f"{W}/.git"):
    sh(f"git config --global --add safe.directory {W}")
    print(sh(f"cd {W} && git fetch -q; git log --oneline -3; "
             "echo '--- uncommitted:'; git status --short | head -20; "
             "echo '--- on server, not on GitHub:'; git log --oneline origin/main..HEAD; "
             "echo '--- on GitHub, not on server:'; git log --oneline HEAD..origin/main | head -5"))

In [ ]:
# 2. GET THE 25 NOV FILES (pull or clone) — run after you pushed from the Mac
assert not sh("pgrep -f '[r]un_models'"), "A sweep is running: wait for logs/DONE before pulling."

if not os.path.isdir(f"{W}/.git"):
    print(sh(f"git clone -q {REPO} {W}") or "cloned")
else:
    ahead = sh(f"cd {W} && git log --oneline origin/main..HEAD")
    dirty = sh(f"cd {W} && git status --short")
    if ahead or dirty:
        raise SystemExit("STOP — the server has work not on GitHub. Paste this output in the chat:\n"
                         + ahead + "\n" + dirty)
    print(sh(f"cd {W} && git pull --no-rebase"))

if not os.path.lexists(f"{W}/.venv"):          # the repo already carries this link
    os.symlink(VENV, f"{W}/.venv")
print(".venv ->", os.path.realpath(f"{W}/.venv"), "| works:", os.path.isfile(f"{W}/.venv/bin/python"))

for f in ["scripts/human_agreement.py", "docs/runbook_human_agreement.md",
          "docs/annotation/guideline_insurance_v2.0.md",
          "data/processed/checklists/insurance_signed.json"]:
    print(("OK       " if os.path.exists(f"{W}/{f}") else "MISSING  ") + f)
print(sh(f"cd {W} && {PY} -m pytest -q tests/test_human_agreement.py 2>&1 | tail -1"))

In [ ]:
# 3. NHANES: download (~40 MB), check, decode into applicants
print(sh(f"cd {W} && {PY} scripts/fetch_nhanes.py && {PY} scripts/fetch_nhanes.py --verify", t=3600)[-2000:])
print(sh(f"cd {W} && {PY} scripts/import_nhanes.py", t=3600)[-3000:])

A = f"{W}/data/records/nhanes/applicants.jsonl"
if os.path.exists(A):
    print("\napplicants:", sum(1 for _ in open(A)),
          "| sha256 (goes into PREREGISTRATION.md):", hashlib.sha256(open(A, "rb").read()).hexdigest())
else:
    print("\nNO applicants.jsonl — import failed; paste the output in the chat")

In [ ]:
# 4. THE FROZEN INSURANCE PROBE SET (14,825 per model)
assert os.path.exists(f"{W}/data/processed/checklists/insurance_signed.json"), \
    "Sign the insurance checklist on the Mac, push, then rerun cell 2."
out = sh(f"cd {W} && {PY} scripts/generate_probes.py insurance")
print(out[-2500:])
assert "life_insurance_nhanes" in out, "STOP: not the NHANES pack (cell 3 did not finish)"
assert "explanation_request=76" in out, "STOP: unexpected transparency count"
print("\nOK: NHANES pack, 152 explanation prompts")

In [ ]:
# 5. EXPLANATION PROMPTS ONLY (4 models x 152) — the first insurance answers
PREREG_SIGNED_OR_APPROVED = False      # set True only when signed / professor's OK is in writing
assert PREREG_SIGNED_OR_APPROVED, "Get the pre-registration signed (or the written OK) first."
assert not sh("pgrep -f '[r]un_models'"), "A sweep is already running."

os.makedirs(f"{W}/logs", exist_ok=True)
LOG, DONE = f"{W}/logs/runs.log", f"{W}/logs/DONE"
if os.path.exists(LOG):                           # keep old logs, start a clean one
    os.rename(LOG, f"{W}/logs/runs_{datetime.datetime.now():%Y%m%d_%H%M%S}.log")

env = {**ENV, "HF_TOKEN": getpass.getpass("HF token: "), "GIT_TOKEN": getpass.getpass("GitHub token: ")}
subprocess.Popen(["nohup", "bash", "scripts/run_models.sh", "insurance", "--dimension", "transparency"],
                 stdout=open(LOG, "a"), stderr=subprocess.STDOUT,
                 start_new_session=True, cwd=W, env=env)
print("launched — run cell 6 to watch")

In [ ]:
# 6. WATCH (stops by itself; safe to rerun; the run continues if you stop this cell)
LOG, DONE = f"{W}/logs/runs.log", f"{W}/logs/DONE"
start = time.time()
while time.time() - start < 2 * 3600:
    alive  = sh("pgrep -f '[r]un_models.sh'")
    pushed = sh(f"grep -c PUSHED {LOG} 2>/dev/null") or "0"
    gpu    = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader 2>/dev/null")
    last   = sh(f"grep -E 'START|PUSHED|FAILED|requested' {LOG} 2>/dev/null | tail -1")[:80]
    print(f"[{time.strftime('%H:%M:%S')}] pushed {pushed}/4 | gpu {gpu} | "
          f"{'alive' if alive else 'STOPPED'} | {last}", flush=True)
    if os.path.exists(DONE):
        print("\nDONE:", open(DONE).read().strip()); print(sh(f"grep -E 'requested|PUSHED|FAILED' {LOG}")); break
    if not alive:
        print("\nstopped without DONE — last lines:\n" + sh(f"tail -25 {LOG}")); break
    time.sleep(60)

In [ ]:
# 7. CHECK THE ANSWERS (expect 152 per model, 0 errors)
import collections, json
R = f"{W}/data/processed/runs/insurance/responses.jsonl"
rows = [json.loads(l) for l in open(R)]
print("answers per model:", dict(collections.Counter(r["model_id"] for r in rows)))
print("errors           :", sum(1 for r in rows if r.get("error")))
print("empty answers    :", sum(1 for r in rows if not (r.get("response") or "").strip()))

In [ ]:
# 8. DRAW THE SAMPLE AND BUILD THE RATER PACKETS, THEN PUSH
print(sh(f"cd {W} && {PY} scripts/human_agreement.py export insurance"))
url = f"https://prasadwatane:{getpass.getpass('GitHub token: ')}@github.com/prasadwatane/CASE-STUDY.git"
print(sh(f"cd {W} && git add -A && "
         f"git -c user.email=grail@local -c user.name=grail commit -q -m 'Insurance explanations and CAL1 rater packets'; "
         f"git push {url} main 2>&1 | tail -2"))

## Next, on the Mac

```bash
cd ~/Downloads/grail-audit && git pull
open data/processed/annotation/insurance/      # the HTML packets are here
```

1. Arbiter: `calibration_arbiter_key.html` → returns `calibration_arbiter_key.csv`
2. Raters: `calibration_rater_A.html` / `_B.html` → `python3 scripts/human_agreement.py calibration insurance` (18 of 20)
3. Raters: `rater_A.html` / `rater_B.html` (240 each) → `python3 scripts/human_agreement.py score insurance` → `agreement.md`
4. Arbiter: `arbiter.html` → `python3 scripts/human_agreement.py reference insurance`

Send raters the HTML file plus `GRAIL_Rater_Guideline_v2.0.pdf` and `GRAIL_Rater_Instructions.pdf`.
Never send `KEY_do_not_open_until_scored.json`.